# IDX — Colab Training Adapter (thin)

**Roles**
- GitHub Actions = daily operational signals (`ops_sma_v0`)
- **Colab = heavy/weekend ML training & research**
- Telegram = output only
- Paper portfolio = measurement only

**Safety**
- No auto-promotion
- Production pointer unchanged
- Economic edge remains **UNVERIFIED**
- GPU optional — CPU always works
- Training budget default **1200s** (`COLAB_TRAINING_BUDGET_SEC`)

Open from GitHub: use *Open in Colab* badge in `colab/README.md`.

> Notebook is a thin adapter. All ML/Governor/shadow logic lives under `src/python/`.


In [ ]:
# 1) Clone / checkout (Python-safe — no broken $REF shell expansion)
import os, sys, subprocess
from pathlib import Path

REPO = os.environ.get("IDX_REPO", "https://github.com/whatman42/idx.git")
REF = os.environ.get("IDX_REF", "main")
WORKDIR = Path("/content/idx") if Path("/content").exists() else Path.cwd() / "idx_colab_workspace"

def _run(cmd, cwd=None):
    print("+", " ".join(cmd))
    subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)

if not WORKDIR.exists():
    WORKDIR.parent.mkdir(parents=True, exist_ok=True)
    _run(["git", "clone", "--depth", "1", "-b", REF, REPO, str(WORKDIR)])
else:
    _run(["git", "fetch", "--depth", "1", "origin", REF], cwd=WORKDIR)
    _run(["git", "checkout", REF], cwd=WORKDIR)
    try:
        _run(["git", "pull", "--ff-only", "origin", REF], cwd=WORKDIR)
    except subprocess.CalledProcessError as e:
        print("pull skipped/failed:", e)

os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))
print("cwd", os.getcwd())
print("ref", REF, "repo", REPO)


In [ ]:
# 2) Install deps (CPU-safe; GPU libs optional)
import subprocess, sys
from pathlib import Path

def pip_install(req):
    p = Path(req)
    if p.exists():
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(p)])
        print("installed", p)
    else:
        print("skip missing", p)

pip_install("requirements.txt")
pip_install("requirements-colab.txt")
print("python", sys.version)


In [ ]:
# 3) Hardware probe
import os
from src.python.colab.hardware import probe_hardware, benchmark_lgbm_cpu_gpu

budget = float(os.getenv("COLAB_TRAINING_BUDGET_SEC", "1200"))
hw = probe_hardware(budget_sec=budget, try_lgbm_gpu=False)
print(hw.print_summary())
print(hw.to_dict())


In [ ]:
# 4) Optional CPU vs GPU benchmark (skipped automatically if no GPU)
bench = None
if hw.cuda_available:
    bench = benchmark_lgbm_cpu_gpu()
    print(bench)
else:
    print({"backend": "cpu", "reason": "no_gpu"})


In [ ]:
# 5) Run governed training via repository entrypoint (NOT notebook-local ML)
import os
from src.python.colab.run_training import run_colab_training

report = run_colab_training(
    budget_sec=float(os.getenv("COLAB_TRAINING_BUDGET_SEC", "1200")),
    out_dir="models/candidates",
    report_dir="artifacts/training",
    promote=False,          # hard-rejected inside orchestrator even if True
    run_shadow=True,
    try_gpu_benchmark=True,
)
print("status:", report.get("status"))
print("promoted:", report.get("promoted"))
print("pointer_unchanged:", report.get("production_pointer_unchanged"))
print("edge:", report.get("economic_edge"))
print("live_execution:", report.get("live_execution"))
print("signal_only:", report.get("signal_only"))
print("models:", [m.get("model_id") for m in report.get("models") or []])


In [ ]:
# 6) Safety asserts — training must never promote or enable live execution
assert report.get("promoted") is False, "Colab must never promote"
assert report.get("production_pointer_unchanged") is True, "Production pointer must stay unchanged"
assert report.get("economic_edge") == "UNVERIFIED"
assert report.get("live_execution") is False
assert report.get("signal_only") is True
assert report.get("promotion", {}).get("decision") == "REJECT"
print("SAFETY_ASSERTS_PASS")


In [ ]:
# 7) Optional: publish reports to GitHub (requires Colab secret GH_PAT)
# Secrets → GH_PAT (repo scope). Training works WITHOUT this secret.
# Never put tokens in notebook source.
from src.python.colab.publish import publish_candidates

pub = publish_candidates(
    paths=[
        "artifacts/training/last_training_report.json",
        "artifacts/training/shadow_report.json",
    ]
)
safe = {k: v for k, v in pub.items() if k not in ("token",) and "token" not in k.lower()}
print(safe)
assert pub.get("production_pointer_touched") is False
assert pub.get("force_push") is False
print("PUBLISH_SAFE")
